In [64]:
%pip install uv --quiet
!uv pip install pandas numpy plotly matplotlib
!uv sync
!{sys.executable} -m pip install -U kaleido

Note: you may need to restart the kernel to use updated packages.


Using Python 3.12.3 environment at: C:\Users\logan\OneDrive\Documents\SeniorSpring\Adv Data Sci\Modern-Store-Of-Value\.venv
Checked 4 packages in 13ms
Resolved 147 packages in 3ms
Checked 143 packages in 16ms


In [65]:
# Data manipulation tools
import pandas as pd
import datetime
import time
import numpy as np

# Visualization tools
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# OS tools
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Custom Stooq data processor
from scripts.stooq_processor import StooqProcessor

In [66]:
stooq_tickers = {
    "Crypto ETFs": [
        "BITW",  # Bitwise 10 Crypto Index
        "IBIT",  # iShares Bitcoin Trust (Replaces BTC-USD)
        "ETHA"   # iShares Ethereum Trust (Replaces ETH-USD)
    ], 
    
    "Individual Stocks": [
        "NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"
    ],
    
    "Sector ETFs": [
        "XLU"    # Utilities Select Sector SPDR Fund
    ],
    
    "Broad Market ETFs": [
        "SPY",   # S&P 500
        "VTI"   # Total US Market (replaces Wilshire 5000)
    ],
    
    "Commodity ETFs (Metals)": [
        "GLD",   # Gold (Baseline)
        "SLV",   # Silver
        "PPLT",  # Platinum
        "PALL"   # Palladium
    ],
    
    "Commodity ETFs (Agriculture)": [
        "WEAT",  # Wheat
        "SOYB",  # Soybeans
        "DBA"    # Broad Agriculture
    ]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [67]:
# -----------------------------
# Flatten tickers + category map
# -----------------------------
category_map = {
    ticker: category
    for category, tickers in stooq_tickers.items()
    for ticker in tickers
}

flat_tickers = list(category_map.keys())


# -----------------------------
# Download data
# -----------------------------
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:

    # Stores valid tickers
    valid_tickers = [
        t for t in flat_tickers
        if processor.has_ticker(t)
    ]

    # Prints tickers that were not valid
    missing = sorted(set(flat_tickers) - set(valid_tickers))
    if missing:
        print("Skipping missing tickers:", missing)

    data = processor.download(
        valid_tickers,
        start=start_date,
        end=end_date,
    )


# -----------------------------
# Attach metadata
# -----------------------------
for ticker, frame in data.items():
    data[ticker] = frame.assign(
        Ticker=ticker,
        Category=category_map[ticker],
    )


# -----------------------------
# Combine dataset
# -----------------------------
combined_data = pd.concat(data.values()).reset_index()

# data['AAPL'].tail()
# data["BITW"]
# combined_data[combined_data['Ticker'] == 'BITW']
print(combined_data)
# combined_data

           Date     Open     High      Low    Close       Volume  OpenInt  \
0    2025-12-31  62.8200  63.1200  56.5600  58.7600    1752680.0        0   
1    2026-01-31  59.9400  66.4800  54.5000  55.6601    2425192.0        0   
2    2026-02-28  51.2878  52.2950  40.6599  43.0400    4596906.0        0   
3    2026-03-26  42.9952  49.4500  42.9952  44.8700    1583267.0        0   
4    2024-01-31  26.4000  26.4100  22.0200  24.3000  207876989.0        0   
...         ...      ...      ...      ...      ...          ...      ...   
1307 2025-11-30  26.6000  26.9400  25.5500  26.4200    3917732.0        0   
1308 2025-12-31  26.3600  26.6666  25.4000  25.5200    4877035.0        0   
1309 2026-01-31  25.5000  26.0500  25.4250  25.6600    5404587.0        0   
1310 2026-02-28  25.5500  26.1500  25.5400  26.0200    5422359.0        0   
1311 2026-03-26  26.0400  27.1429  25.8600  27.1100   37145895.0        0   

     Ticker                      Category  
0      BITW                   C

## Metric - Calmar

### Calculate

In [68]:
"""
Context: 
- Basically Calmar says for every 1% of pain or downturn, how much return do we see on average. 
- So 1 would be breaking even in a sense, and 2% would be for every 1% we see 2% annualized growth.
"""

'\nContext: \n- Basically Calmar says for every 1% of pain or downturn, how much return do we see on average. \n- So 1 would be breaking even in a sense, and 2% would be for every 1% we see 2% annualized growth.\n'

In [69]:
# Ensure Date is a datetime object for accurate time-delta calculations
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

# Sort by ticker and date
combined_data = combined_data.sort_values(by=['Ticker', 'Date'])

# Calculate rolling peak / maximum with groupby and cummax
combined_data['Peak'] = combined_data.groupby('Ticker')['Close'].cummax()

# Calculate percentage drawdowns from peak
combined_data['Drawdown'] = (combined_data['Close'] - combined_data['Peak']) / combined_data['Peak']

# Aggregate the necessary metrics per Ticker and Category
agg_df = combined_data.groupby(['Category', 'Ticker']).agg(
    Max_Drawdown=('Drawdown', lambda x: abs(x.min())), # Absolute value
    First_Price=('Close', 'first'),
    Last_Price=('Close', 'last'),
    First_Date=('Date', 'min'),
    Last_Date=('Date', 'max')
).reset_index()

# Calculate exact years elapsed for each ticker
agg_df['Years'] = (agg_df['Last_Date'] - agg_df['First_Date']).dt.days / 365.25

# Calculate Compound Annual Growth Rate (CAGR)
agg_df['CAGR'] = (agg_df['Last_Price'] / agg_df['First_Price']) ** (1 / agg_df['Years']) - 1

# Calculate raw Calmar Ratio
agg_df['Calmar_Ratio'] = agg_df['CAGR'] / agg_df['Max_Drawdown']

# Drop NaNs
calmar_df = agg_df.dropna(subset=['Calmar_Ratio']).copy()


# ── NEW: Percentile Normalization (1 to 10 Scale) ────────────────────────
# Calculate the exact percentile rank of each asset (0.0 to 1.0)
pct_rank = calmar_df['Calmar_Ratio'].rank(pct=True)

min_rank = pct_rank.min()
max_rank = pct_rank.max()

# Map the percentiles smoothly to a continuous 1.0 - 10.0 scale
if max_rank == min_rank:
    calmar_df['Score_1_10'] = 10.0
else:
    calmar_df['Score_1_10'] = (
        1 + 9 * (pct_rank - min_rank) / (max_rank - min_rank)
    ).round(2)

# Sort from best to worst based on the new Score
calmar_df = calmar_df.sort_values(by='Score_1_10', ascending=False)

# Formatting for hover data
calmar_df['Score_Formatted'] = calmar_df['Score_1_10'].astype(str)
calmar_df['Calmar_Formatted'] = calmar_df['Calmar_Ratio'].round(2).astype(str)
calmar_df['CAGR_Formatted'] = (calmar_df['CAGR'] * 100).round(2).astype(str) + '%'
calmar_df['Max_Drawdown_Formatted'] = (calmar_df['Max_Drawdown'] * -100).round(2).astype(str) + '%'

# Print results
print(calmar_df[['Ticker', 'Category', 'Score_1_10', 'Calmar_Ratio']])

   Ticker                      Category  Score_1_10  Calmar_Ratio
21    WMT             Individual Stocks       10.00      1.065618
19   NVDA             Individual Stocks        9.59      1.036679
5     GLD       Commodity ETFs (Metals)        9.18      1.034509
2     DBA  Commodity ETFs (Agriculture)        8.77      1.011946
22    XLU                   Sector ETFs        8.36      0.614858
11   IBIT                   Crypto ETFs        7.95      0.554481
12   AAPL             Individual Stocks        7.55      0.532745
0     SPY             Broad Market ETFs        7.14      0.531338
8     SLV       Commodity ETFs (Metals)        6.73      0.523674
16    JNJ             Individual Stocks        6.32      0.468394
1     VTI             Broad Market ETFs        5.91      0.431426
7    PPLT       Commodity ETFs (Metals)        5.50      0.337701
18   MSFT             Individual Stocks        5.09      0.317976
13    AMD             Individual Stocks        4.68      0.295481
17    LOW 

In [70]:
"""
BASELINE: 
- GOLD: -17.62%. With that being said we would like something to be 17% or better

WINNERS: 
- DBA (Broad Agriculture): At -10.15%, it had the lowest downturn of any asset. It acted as an incredibly stable store of value.

- XLU (Utilities): At -17.63%, it practically perfectly matches Gold's downside protection. This completely validates the finding mentioned in your report that XLU possesses strong defensive characteristics.

- JNJ & WMT (Consumer Staples): With drawdowns of -18.77% and -20.24%, these defensive stocks outperformed the broader market (SPY at -23.92%) and proved highly resilient during downturns.

LOSERS: 
- Palladium (PALL), Tesla (TSLA), Wheat (WEAT), Nvidia (NVDA), and AMD
  - All of these assets lost over 60% of their value from their peak at some point. 
  - Even though assets like NVDA have massive long-term returns, a 62% drawdown means an investor could lose 
    more than half their wealth in a crisis. They are speculative growth assets, not stores of value.
    
CRYPTOCURRENCIES:
- ETHA (Ethereum) and IBIT (Bitcoin): 
  - Experienced severe crashes of -55.76% and -43.92%. They failed the stability test and behave much more 
    like high-growth tech stocks than "digital gold."

- BITW (Crypto Index): 
  - The diversification of the index helped soften the blow to -26.75%, but it still suffered worse drawdowns
    than the broader stock market (SPY/VTI). Crypto broadly failed to preserve capital during market corrections.
"""


'\nBASELINE: \n- GOLD: -17.62%. With that being said we would like something to be 17% or better\n\nWINNERS: \n- DBA (Broad Agriculture): At -10.15%, it had the lowest downturn of any asset. It acted as an incredibly stable store of value.\n\n- XLU (Utilities): At -17.63%, it practically perfectly matches Gold\'s downside protection. This completely validates the finding mentioned in your report that XLU possesses strong defensive characteristics.\n\n- JNJ & WMT (Consumer Staples): With drawdowns of -18.77% and -20.24%, these defensive stocks outperformed the broader market (SPY at -23.92%) and proved highly resilient during downturns.\n\nLOSERS: \n- Palladium (PALL), Tesla (TSLA), Wheat (WEAT), Nvidia (NVDA), and AMD\n  - All of these assets lost over 60% of their value from their peak at some point. \n  - Even though assets like NVDA have massive long-term returns, a 62% drawdown means an investor could lose \n    more than half their wealth in a crisis. They are speculative growth a

### Visualize

In [71]:
# ── Scale Benchmarks using Interpolation ─────────────────────────────────
# Because we are now using relative ranks instead of raw distances, we use 
# numpy's linear interpolation to find exactly where the benchmarks fall on the 1-10 axis.
calmar_df_sorted = calmar_df.sort_values(by='Calmar_Ratio')

try:
    gold_calmar_raw = calmar_df[calmar_df['Ticker'] == 'GLD']['Calmar_Ratio'].values[0]
except IndexError:
    gold_calmar_raw = 0 
    
gold_scaled = np.interp(
    gold_calmar_raw, 
    calmar_df_sorted['Calmar_Ratio'], 
    calmar_df_sorted['Score_1_10']
)

good_scaled = np.interp(
    1.0, 
    calmar_df_sorted['Calmar_Ratio'], 
    calmar_df_sorted['Score_1_10']
)


# ── Build the Bar Chart ──────────────────────────────────────────────────
fig = px.bar(
    calmar_df,
    x="Ticker",
    y="Score_1_10",
    color="Category", 
    title="Risk-Adjusted Performance Score (1-10) — Based on Percentile Ranking (Calmar)",
    labels={"Score_1_10": "Performance Score (10 = Top Percentile)", "Ticker": "Asset"},
    hover_data={
        "Score_Formatted": True,
        "Calmar_Formatted": True, 
        "CAGR_Formatted": True, 
        "Max_Drawdown_Formatted": True, 
        "Score_1_10": False,
        "Category": False
    }
)

# Add the horizontal baseline for Gold
fig.add_hline(
    y=gold_scaled, 
    line_dash="dot", 
    line_color="black",
    line_width=2,
    annotation_text=f"Gold Benchmark (Raw Calmar: {gold_calmar_raw:.2f})",
    annotation_position='top left' 
)

# Add the "Good" benchmark line at 1.0
fig.add_hline(
    y=good_scaled, 
    line_dash="solid", 
    line_color="rgba(128, 128, 128, 0.5)",
    line_width=1,
    annotation_text="Good Risk-Adjusted Return (Raw Calmar: 1.0)",
    annotation_position='top right'
)

# Format axes and template
fig.update_layout(
    xaxis=dict(
        categoryorder="array",
        categoryarray=calmar_df["Ticker"]
    ),
    xaxis_tickangle=-45,
    template="plotly_white",
    yaxis=dict(range=[0, 10.5]) # Add a little headroom so the 10.0 bar isn't flush with the ceiling
)

fig.show()
fig.write_html("../plots/calmar_percentile_score_barchart.html")

## Inflation Adjusted Returns For All

### Fetch FRED Inflation Data

In [72]:
# Load CPI data directly from local CSV
print("Loading CPI data from local CSV...")


# Read the CSV, parse the dates, and set the index
cpi_full = pd.read_csv("../data/CPIAUCSL.csv", parse_dates=['observation_date'], index_col='observation_date')

# Filter the dataset to only include your specific timeframe
cpi_data = cpi_full.loc[start_date:end_date].copy()

# Calculate total inflation and cumulative inflation over time
cpi_start_val = cpi_data['CPIAUCSL'].iloc[0]
cpi_end_val = cpi_data['CPIAUCSL'].iloc[-1]
total_inflation_pct = ((cpi_end_val - cpi_start_val) / cpi_start_val) * 100
cpi_data['Cumulative_Inflation_%'] = ((cpi_data['CPIAUCSL'] - cpi_start_val) / cpi_start_val) * 100

print(f"Total CPI Inflation ({start_date} to {end_date}): {total_inflation_pct:.2f}%\n")

Loading CPI data from local CSV...
Total CPI Inflation (2021-01-01 to 2026-03-31): 24.66%



### Prepare Monthly Data & Calculate Inflation Adjusted Returns

In [73]:
# Prepare monthly data & calculate static returns
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

# Resample monthly frequency, taking last closing price of each month
monthly_data = (
    combined_data.set_index('Date')
    .groupby(['Ticker', 'Category'])
    .resample('MS')['Close']
    .last()
    .reset_index()
)

# Calculate total returns and inflation-adjusted returns for each asset
results = []
for ticker in monthly_data['Ticker'].unique():
    ticker_df = monthly_data[monthly_data['Ticker'] == ticker].sort_values('Date')
    
    # No data in ticker condition
    if ticker_df.empty:
        continue
        
    # Start and end price for total return calculation
    start_price = ticker_df.iloc[0]['Close']
    end_price = ticker_df.iloc[-1]['Close']
    
    # Calculate total return and adjust for inflation
    asset_return_pct = ((end_price - start_price) / start_price) * 100
    real_return_pct = asset_return_pct - total_inflation_pct
    
    # Append results
    results.append({
        'Category': ticker_df.iloc[0]['Category'],
        'Ticker': ticker,
        'Total_Return_%': asset_return_pct,
        'Total_Inflation_%': total_inflation_pct,
        'Inflation_Adjusted_Return_%': real_return_pct
    })

# Format and display the table
returns_df = pd.DataFrame(results).sort_values(by='Inflation_Adjusted_Return_%', ascending=False)
formatted_df = returns_df.copy()
formatted_df['Total_Return_%'] = formatted_df['Total_Return_%'].round(2).astype(str) + '%'
formatted_df['Total_Inflation_%'] = formatted_df['Total_Inflation_%'].round(2).astype(str) + '%'
formatted_df['Inflation_Adjusted_Return_%'] = formatted_df['Inflation_Adjusted_Return_%'].round(2).astype(str) + '%'

display(formatted_df)
formatted_df.to_csv('../data/inflation_adjusted_returns.csv', index=False)



,Category,Ticker,Total_Return_%,Total_Inflation_%,Inflation_Adjusted_Return_%
12,Individual Stocks,NVDA,1221.63%,24.66%,1196.97%
21,Individual Stocks,WMT,173.32%,24.66%,148.66%
15,Commodity ETFs (Metals),SLV,143.18%,24.66%,118.52%
1,Individual Stocks,AMD,137.94%,24.66%,113.28%
6,Commodity ETFs (Metals),GLD,132.11%,24.66%,107.45%
0,Individual Stocks,AAPL,96.87%,24.66%,72.21%
17,Broad Market ETFs,SPY,85.14%,24.66%,60.48%
22,Sector ETFs,XLU,69.87%,24.66%,45.21%
19,Broad Market ETFs,VTI,68.81%,24.66%,44.15%
4,Commodity ETFs (Agriculture),DBA,65.41%,24.66%,40.75%


### Visualize

In [74]:
"""
Function: calc_cumulative_return
Purpose: Calculate cumulative returns over time for each asset, starting from the first month in the dataset.
"""
def calc_cumulative_return(group):
    group = group.sort_values('Date')
    start_price = group['Close'].iloc[0]
    group['Cumulative_Return_%'] = ((group['Close'] - start_price) / start_price) * 100
    return group

# Cumulate returns for each asset
cumulative_data = monthly_data.groupby('Ticker', group_keys=False).apply(calc_cumulative_return)

# Plot
fig = px.line(
    cumulative_data,
    x='Date',
    y='Cumulative_Return_%',
    color='Ticker',
    hover_data=['Category'],
    title=f'Asset Cumulative Returns vs. US Inflation ({start_date} to {end_date})',
    labels={'Cumulative_Return_%': 'Cumulative Return (%)', 'Date': 'Date'}
)

fig.update_traces(visible='legendonly')

inflation_trace = go.Scatter(
    x=cpi_data.index,
    y=cpi_data['Cumulative_Inflation_%'],
    name='Cumulative US Inflation (Baseline)',
    fill='tozeroy',  
    mode='lines',
    line=dict(color='rgba(64, 64, 64, 0.9)', width=4, dash='dot'), 
    fillcolor='rgba(128, 128, 128, 0.25)', 
    hoverinfo='x+y+name'
)
fig.add_trace(inflation_trace)

fig.data = (fig.data[-1],) + fig.data[:-1]

fig.update_layout(
    height=800,
    width=1400,
    template="plotly_white",
    hovermode="closest",
    legend=dict(
        title="<b>Assets</b><br>(Double-click to isolate)",
        itemsizing="constant"
    )
)

fig.show()
fig.write_html("../plots/inflation_adjusted_returns.html")

C:\Users\logan\AppData\Local\Temp\ipykernel_29644\366017513.py:12: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



### MinMax Normalization

In [75]:
import pandas as pd
import numpy as np
import plotly.express as px

# ── 1. Calculate Monthly Returns & Inflation ──────────────────────────────
# Ensure Date is datetime
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

# Resample monthly frequency, taking last closing price of each month
monthly_data = (
    combined_data.set_index('Date')
    .groupby(['Ticker', 'Category'])
    .resample('MS')['Close']
    .last()
    .reset_index()
)

results = []
for ticker in monthly_data['Ticker'].unique():
    ticker_df = monthly_data[monthly_data['Ticker'] == ticker].sort_values('Date')
    
    if ticker_df.empty:
        continue
        
    start_price = ticker_df.iloc[0]['Close']
    end_price = ticker_df.iloc[-1]['Close']
    
    asset_return_pct = ((end_price - start_price) / start_price) * 100
    real_return_pct = asset_return_pct - total_inflation_pct
    
    results.append({
        'Category': ticker_df.iloc[0]['Category'],
        'Ticker': ticker,
        'Total_Return_%': asset_return_pct,
        'Total_Inflation_%': total_inflation_pct,
        'Inflation_Adjusted_Return_%': real_return_pct
    })

returns_df = pd.DataFrame(results)

# ── 2. Apply Percentile Normalization (1-10 Scale) ────────────────────────
# Calculate the exact percentile rank of each asset (0.0 to 1.0)
pct_rank = returns_df['Inflation_Adjusted_Return_%'].rank(pct=True)

min_rank = pct_rank.min()
max_rank = pct_rank.max()

# Map the percentiles smoothly to a continuous 1.0 - 10.0 scale
if max_rank == min_rank:
    returns_df['Inflation_Score_1_10'] = 10.0
else:
    returns_df['Inflation_Score_1_10'] = (
        1 + 9 * (pct_rank - min_rank) / (max_rank - min_rank)
    ).round(2)

score_df = returns_df.sort_values(by='Inflation_Score_1_10', ascending=False).copy()

# Formatting specific columns for clean hover data in the visualization
score_df['Score_Formatted'] = score_df['Inflation_Score_1_10'].astype(str)
score_df['Real_Return_Formatted'] = score_df['Inflation_Adjusted_Return_%'].round(2).astype(str) + '%'
score_df['Total_Return_Formatted'] = score_df['Total_Return_%'].round(2).astype(str) + '%'


# ── 3. Scale the Breakeven Benchmark ──────────────────────────────────────
# Because we are now using relative ranks instead of raw distances, we use 
# numpy's linear interpolation to find exactly where 0.0% falls on our new 1-10 axis.
score_df_sorted_by_return = score_df.sort_values(by='Inflation_Adjusted_Return_%')

breakeven_scaled = np.interp(
    0.0, 
    score_df_sorted_by_return['Inflation_Adjusted_Return_%'], 
    score_df_sorted_by_return['Inflation_Score_1_10']
)


# ── 4. Build the Visualization ────────────────────────────────────────────
fig_score = px.bar(
    score_df,
    x='Ticker',
    y='Inflation_Score_1_10',
    color='Category',
    title='Purchasing Power Protection Score (1-10) — Based on Percentile Ranking',
    labels={'Inflation_Score_1_10': 'Score (10 = Top Percentile)', 'Ticker': 'Asset'},
    hover_data={
        'Score_Formatted': True,
        'Real_Return_Formatted': True,
        'Total_Return_Formatted': True,
        'Inflation_Score_1_10': False,
        'Category': False,
        'Inflation_Adjusted_Return_%': False,
        'Total_Return_%': False,
        'Total_Inflation_%': False
    },
    template='plotly_white'
)

# Add the horizontal baseline for 0% Real Return (Breakeven)
fig_score.add_hline(
    y=breakeven_scaled,
    line_dash="dot",
    line_color="black",
    line_width=2,
    annotation_text=f"Breakeven / 0% Real Return (Score: {breakeven_scaled:.2f})",
    annotation_position='top right' if breakeven_scaled < 8 else 'bottom right' 
)

# Format axes and add headroom to the chart
fig_score.update_layout(
    xaxis=dict(
        categoryorder="array",
        categoryarray=score_df["Ticker"]
    ),
    xaxis_tickangle=-45,
    yaxis=dict(range=[0, 10.5]) 
)

fig_score.show()
fig_score.write_html("../plots/inflation_percentile_score_barchart.html")

## Gold Visualized Over Time Range

In [80]:
import plotly.express as px

# ── 1. Filter the Data ────────────────────────────────────────────────────
gld_data = combined_data[
    (combined_data['Ticker'] == 'GLD') & 
    (combined_data['Date'] >= start_date) & 
    (combined_data['Date'] <= end_date)
].copy()

# Sort by date just to be absolutely sure the line draws sequentially
gld_data = gld_data.sort_values(by='Date')

# ── 2. Build the Visualization ────────────────────────────────────────────
fig = px.line(
    gld_data,
    x='Date',
    y='Close',
    labels={'Close': 'Closing Price (USD)', 'Date': 'Date'},
    template='plotly_white'
)

# Force a pure white background AND increase global font size
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=18) # <--- Your teammate's addition
)

# Optional styling to make the line look sleek and distinct
fig.update_traces(line=dict(color="#1b3a6b", width=2.5)) 

# ── 3. Configure the Manual Download Button ───────────────────────────────
fig.show(config={
    'toImageButtonOptions': {
        'format': 'png', 
        'filename': 'gold_price_poster_ready',
        'height': 600,  
        'width': 1000,  
        'scale': 3      
    }
})